In [1]:
from vncv.ocr import extract_text

results = extract_text("RidomilGold 2.jpg", lang="vi")

print("OCR Results:", results)

[VNCV System] Using ONNX for Vietnamese Recognition
[INFO] Loading Encoder: c:\Users\ASUS\anaconda3\envs\ocr\Lib\site-packages\vncv\weights\model_encoder.onnx
[INFO] Loading Decoder: c:\Users\ASUS\anaconda3\envs\ocr\Lib\site-packages\vncv\weights\model_decoder.onnx
[INFO] Loading Vocab:   c:\Users\ASUS\anaconda3\envs\ocr\Lib\site-packages\vncv\weights\vocab.json

[IO Info]
  Encoder:
    Inputs : [('input', ['batch', 3, 32, 'im_width'], 'tensor(float)')]
    Outputs: [('memory', ['2*((im_width//4))', 1, 256], 'tensor(float)')]
  Decoder:
    Inputs : [('tgt_inp', ['seq_len', 'batch'], 'tensor(int64)'), ('memory', ['feat_len', 'batch', 256], 'tensor(float)')]
    Outputs: [('output', [1, 'seq_len', 233], 'tensor(float)')]

OCR Results: ['14 ngày', 'Trong Cạo sạch vét bệnh, quét dung dịch iên một cạc chiến', 'mus', 'Theo Pho 800g/phuy 200L, phun 5 phuy/hoa', '1', '10', 'và phát triển náng vào mùa ra', 'Phun khi bênh chơm xuất hiện', 'Pha 1000g/phuy 200L, phun 5 phuong', 'TWỤ, CÁCH NHÂU',

In [ ]:
from pydantic_ai.models.openrouter import OpenRouterModel
from pydantic_ai.providers.openrouter import OpenRouterProvider
import os
import dotenv

dotenv.load_dotenv()

def _get_open_router_client():
    api_key = os.getenv("OPENROUTER_API_KEY", "")
    if not api_key:
        raise ValueError("Không tìm thấy OpenRouter API Key.")
    provider = OpenRouterProvider(api_key=api_key)
    model = OpenRouterModel('openai/gpt-oss-120b:free', provider=provider)
    return model

In [4]:
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider

def _get_gemini_client():
    api_key = os.getenv("GEMINI_API_KEY", "")
    if not api_key:
        raise ValueError("Không tìm thấy Gemini API Key.")
    provider = GoogleProvider(api_key=api_key)
    model = GoogleModel('gemini-2.5-flash', provider=provider)
    return model

In [5]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from typing import List, Optional
from datetime import date

async def extract_product_info(transcript_text: str) -> dict:
    """Trích xuất thông tin sản phẩm từ OCR."""

    class ActiveIngredient(BaseModel):
        name: str = Field(..., description="Tên hoạt chất")
        content: str = Field(..., description="Hàm lượng hoạt chất")


    class ProductInfo(BaseModel):
        product_name: str = Field(..., description="Tên sản phẩm")
        product_type: str = Field(..., description="Loại sản phẩm")
        manufacturer: str = Field(..., description="Nhà sản xuất")
        registration_number: Optional[str] = Field(
            None,
            description="Số đăng ký"
        )

        active_ingredients: List[ActiveIngredient] = Field(
            ...,
            description="Danh sách hoạt chất"
        )

        dosage: Optional[str] = Field(
            None,
            description="Liều lượng sử dụng"
        )

        target_crops: Optional[List[str]] = Field(
            None,
            description="Danh sách cây trồng áp dụng"
        )

        target_pests: Optional[List[str]] = Field(
            None,
            description="Danh sách bệnh/dịch hại"
        )

        pre_harvest_interval_days: Optional[int] = Field(
            None,
            description="Thời gian cách ly trước thu hoạch (ngày)"
        )

        expiry_date: Optional[date] = Field(
            None,
            description="Ngày hết hạn"
        )

        confidence_score: float = Field(
            ...,
            ge=0.0,
            le=1.0,
            description="Độ tin cậy OCR/trích xuất"
        )

    model = _get_gemini_client()
    agent = Agent(model, output_type=ProductInfo)
    prompt = (
        "Dựa vào đoạn OCR text sau, hãy trích xuất thông tin sản phẩm. OCR có thể chứa lỗi, hãy cố gắng trích xuất thông tin chính xác nhất có thể và đánh giá độ tin cậy của thông tin đó. Những field không chỉ rõ trong OCR có thể để trống hoặc null.\n\n"
        "Trả về JSON với product_name (str), product_type (str), manufacturer (str), registration_number (str), active_ingredients (List[ActiveIngredient]), dosage (str), target_crops (List[str]), target_pests (List[str]), pre_harvest_interval_days (int), expiry_date (date), và confidence_score (float).\n\n"
        f"OCR Text:\n{transcript_text[:3000]}"
    )
    result = await agent.run(prompt)
    return result.output.model_dump()

In [6]:
a = ["a", "b", "c"]
print(" ".join(a))

a b c


In [7]:
product_info = await extract_product_info(" ".join(results))

In [8]:
product_info

{'product_name': 'RidomilGold 68 WG',
 'product_type': 'THUỐC TRỪ BỆNH',
 'manufacturer': 'Syngenta',
 'registration_number': 'VN305180',
 'active_ingredients': [{'name': 'Metalaxyl M', 'content': '40 g/kg'},
  {'name': 'Mancozeb', 'content': '640 g/kg'}],
 'dosage': None,
 'target_crops': ['ca cao',
  'điều',
  'cao su',
  'thuốc lá',
  'lạc',
  'hồ tiêu',
  'ngô (bắp)'],
 'target_pests': ['bệnh sương mai',
  'thán thư',
  'loét sọc mặt cạo',
  'chết cây con',
  'chết nhanh',
  'đốm lá'],
 'pre_harvest_interval_days': None,
 'expiry_date': None,
 'confidence_score': 0.8}